# FraudGuard AI — Análise Exploratória de Dados (EDA) e Comparação de Algoritmos
**Contexto:** Mini Capstone — Pós-Graduação em Engenharia de Inteligência Artificial

Este notebook documenta:
1. **Auditoria de Qualidade do Dataset (`creditcard_2023.csv`)**: reconciliação de 568.630 registros, 31 colunas, ausência de valores nulos, distribuição de classes (50%/50%) e auditoria de duplicatas.
2. **Cuidados Metodológicos e Prevenção de Vazamento (*Data Leakage*)**: exclusão estrita de `id` e `Class` da matriz de atributos `X` (`FEATURES = ["V1", ..., "V28", "Amount"]`).
3. **Particionamento Estratificado Reprodutível**: 70% Treino, 15% Validação, 15% Teste (`random_state=42`).
4. **Comparação Justa de Algoritmos**: `Pipeline(StandardScaler, LogisticRegression)` (Baseline) vs `Pipeline(StandardScaler, RandomForestClassifier)` (Candidato).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from ml.src.ingestion import validate_csv_in_chunks
from ml.src.preprocessing import FEATURES, stratified_train_val_test_split
from ml.src.training import run_experiments

In [ ]:
csv_path = PROJECT_ROOT / "creditcard_2023.csv"
summary = validate_csv_in_chunks(csv_path, chunksize=20_000)
print("Resumo de Validação em Chunks:")
print(summary)

In [ ]:
df = pd.read_csv(csv_path)
split = stratified_train_val_test_split(df, random_state=42)
split.quality_report

In [ ]:
artifacts = run_experiments(
    csv_path=csv_path,
    output_dir=PROJECT_ROOT / "ml" / "artifacts",
    parquet_path=PROJECT_ROOT / "ml" / "data" / "creditcard_2023.parquet",
    random_state=42,
)
for version, art in artifacts.items():
    print(f"=== Modelo {version} ({art.algorithm}) ===")
    print("Threshold selecionado (Validação):", art.threshold)
    print("Métricas de Teste Intocado (15%):", art.metrics["test"])